# Maskable PPO Algorithm

**Project:** Maskable PPO  
**Role:** Maskable PPO Algorithm explanation


## 1. PPO overview

PPO is an **on-policy actor-critic** reinforcement learning algorithm. The agent repeatedly performs the following cycle:

1. Observe the current state $s_t$.
2. Use the policy $\pi_\theta(a\mid s_t)$ to select an action $a_t$.
3. Receive a reward $r_t$ and the next state from the environment.
4. Collect a rollout containing states, actions, rewards, log-probabilities, and value estimates.
5. Update the policy while limiting how much it can change in one training iteration.

The **actor** represents the policy and produces an action distribution. The **critic** estimates the value function $V(s)$, which predicts the expected future return from a state.

For Taxi-v4, the actor produces scores for six actions: South, North, East, West, Pickup, and Dropoff. Standard PPO can assign probability to all six actions. Maskable PPO removes invalid actions before sampling from this distribution.


## 2. Advantage estimation

The advantage tells us whether an action produced a better or worse result than the critic expected:

$$A_t = Q(s_t,a_t) - V(s_t)$$

A positive advantage means that the selected action was better than expected, so PPO should increase its probability. A negative advantage means that it was worse than expected, so PPO should decrease its probability.

A simple one-step estimate is the temporal-difference residual:

$$A_t \approx r_t + \gamma V(s_{t+1}) - V(s_t)$$

In practice, PPO commonly uses Generalized Advantage Estimation (GAE), which combines information from multiple future steps. The one-step calculation below is sufficient to illustrate the meaning of advantage.


In [7]:
# A Taxi-v4 movement normally gives reward -1.
# The action may still have a positive advantage if it moves the taxi
# to a state whose estimated future return is better.
reward = -1.0
current_value = -4.0
next_value = -2.5
gamma = 0.99

advantage = reward + gamma * next_value - current_value

print(f"One-step advantage: {advantage:.3f}")
print("Interpretation: the action was better than the critic expected.")

assert advantage > 0


One-step advantage: 0.525
Interpretation: the action was better than the critic expected.


## 3. Probability ratio and clipping

PPO compares the probability of the selected action under the current policy with its probability under the policy that collected the rollout:

$$r_t(\theta) = \frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{old}}(a_t\mid s_t)}$$

- $r_t = 1$ means the action probability did not change.
- $r_t > 1$ means the new policy makes the action more likely.
- $r_t < 1$ means the new policy makes the action less likely.

A large policy change may make learning unstable. PPO therefore uses the clipped surrogate objective:

$$L^{CLIP}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta)A_t,\; \operatorname{clip}(r_t(\theta),1-\epsilon,1+\epsilon)A_t\right)\right]$$

The common value $\epsilon=0.2$ limits the useful probability ratio to the interval $[0.8,1.2]$. Clipping does not directly force every ratio into this interval; it removes the incentive to move the policy farther when that change would improve the surrogate objective too aggressively.


## 4. Invalid actions in Taxi-v4

Taxi-v4 has six discrete actions. For the state used in the previous notebooks, the environment returns the following mask:

```python
action_mask = [1, 1, 0, 1, 0, 0]
```

| ID | Action | Mask | Status |
|---:|---|---:|---|
| 0 | South | 1 | Valid |
| 1 | North | 1 | Valid |
| 2 | East | 0 | Invalid |
| 3 | West | 1 | Valid |
| 4 | Pickup | 0 | Invalid |
| 5 | Dropoff | 0 | Invalid |

A standard PPO policy may initially assign probability to all actions. Maskable PPO sets the logits of invalid actions to a very large negative value before applying softmax. Their probabilities therefore become zero, and the remaining probability mass is redistributed across valid actions.


In [8]:
import numpy as np

ACTION_NAMES = np.array([
    "South", "North", "East", "West", "Pickup", "Dropoff"
])

logits = np.array([1.2, 0.8, 1.5, 0.2, 1.0, -0.5])
action_mask = np.array([1, 1, 0, 1, 0, 0], dtype=bool)

def softmax(values):
    finite_values = values[np.isfinite(values)]
    shifted = values - np.max(finite_values)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum()

standard_probabilities = softmax(logits)
masked_logits = np.where(action_mask, logits, -np.inf)
masked_probabilities = softmax(masked_logits)

print(f"{'Action':<8} {'Standard PPO':>14} {'Maskable PPO':>14}")
for name, standard_p, masked_p in zip(
    ACTION_NAMES, standard_probabilities, masked_probabilities
):
    print(f"{name:<8} {standard_p:>14.3f} {masked_p:>14.3f}")

assert np.all(masked_probabilities[~action_mask] == 0.0)
assert np.isclose(masked_probabilities.sum(), 1.0)


Action     Standard PPO   Maskable PPO
South             0.228          0.491
North             0.153          0.329
East              0.308          0.000
West              0.084          0.180
Pickup            0.187          0.000
Dropoff           0.042          0.000


## 5. Where Maskable PPO applies the mask

The mask is applied **inside the policy's action distribution**, after the neural network produces action logits and before an action is sampled or selected:

```text
Taxi state
    ↓
Policy network produces 6 logits
    ↓
Current Taxi-v4 action mask is applied
    ↓
Softmax creates a distribution over valid actions only
    ↓
Action is sampled and executed in the environment
```

The action masks must also be stored with rollout data and reused during PPO optimization. This ensures that both the old and current action log-probabilities are calculated using the valid-action distribution for the corresponding state.

Action masking does **not** change Taxi-v4's rewards, transitions, observation space, or PPO clipping rule. It changes which actions are available to the policy distribution.


## 6. Standard PPO versus Maskable PPO

| Component | Standard PPO | Maskable PPO |
|---|---|---|
| Policy network | Produces logits for all actions | Produces logits for all actions |
| Action distribution | Includes every action | Includes only actions allowed by the current mask |
| Invalid action probability | May be greater than zero | Zero |
| Advantage estimation | PPO/GAE | Same as PPO/GAE |
| Probability ratio | Uses ordinary action probabilities | Uses masked action probabilities |
| Clipped objective | PPO clipped objective | Same PPO clipped objective |

Therefore, Maskable PPO is not a different policy-update rule. It is PPO with a state-dependent valid-action distribution used consistently during rollout collection and policy optimization.


## 7. Numerical policy-update example

Consider the Taxi-v4 mask $[1,1,0,1,0,0]$. Suppose the agent selected **South** (action 0), which is valid. The old and current policies below are already normalized over valid actions:

$$\pi_{old} = [0.40, 0.35, 0, 0.25, 0, 0]$$

$$\pi_{new} = [0.50, 0.30, 0, 0.20, 0, 0]$$

For the selected action, the probability ratio is $0.50/0.40=1.25$. If its advantage is $A_t=2.0$ and $\epsilon=0.2$, the clipped ratio is $1.2$. The following cell performs the full calculation.


In [9]:
old_policy = np.array([0.40, 0.35, 0.00, 0.25, 0.00, 0.00])
new_policy = np.array([0.50, 0.30, 0.00, 0.20, 0.00, 0.00])
action_mask = np.array([1, 1, 0, 1, 0, 0], dtype=bool)

selected_action = 0  # South
advantage = 2.0
epsilon = 0.2

assert action_mask[selected_action], "The selected action must be valid"
assert np.all(old_policy[~action_mask] == 0.0)
assert np.all(new_policy[~action_mask] == 0.0)
assert np.isclose(old_policy.sum(), 1.0)
assert np.isclose(new_policy.sum(), 1.0)

old_probability = old_policy[selected_action]
new_probability = new_policy[selected_action]
probability_ratio = new_probability / old_probability
clipped_ratio = np.clip(
    probability_ratio,
    1.0 - epsilon,
    1.0 + epsilon,
)

unclipped_objective = probability_ratio * advantage
clipped_objective = clipped_ratio * advantage
ppo_objective = min(unclipped_objective, clipped_objective)
policy_loss = -ppo_objective  # Optimizers minimize a loss

print(f"Selected action:       {ACTION_NAMES[selected_action]}")
print(f"Old probability:       {old_probability:.2f}")
print(f"New probability:       {new_probability:.2f}")
print(f"Probability ratio:      {probability_ratio:.2f}")
print(f"Clipped ratio:          {clipped_ratio:.2f}")
print(f"Unclipped objective:    {unclipped_objective:.2f}")
print(f"Clipped objective:      {clipped_objective:.2f}")
print(f"Final PPO objective:    {ppo_objective:.2f}")
print(f"Policy loss:           {policy_loss:.2f}")


Selected action:       South
Old probability:       0.40
New probability:       0.50
Probability ratio:      1.25
Clipped ratio:          1.20
Unclipped objective:    2.50
Clipped objective:      2.40
Final PPO objective:    2.40
Policy loss:           -2.40


### Interpretation

The selected action has a positive advantage, so increasing its probability is desirable. However, its probability increased by 25%, giving a ratio of $1.25$. With $\epsilon=0.2$, PPO uses the clipped contribution based on $1.2$ and prevents this sample from rewarding an excessively large policy change.

The invalid actions remain at probability zero throughout the calculation. The neural-network optimizer would minimize the negative objective together with the value-function and entropy terms used by the full implementation.


## References

- Schulman et al., [Proximal Policy Optimization Algorithms](https://arxiv.org/abs/1707.06347)
- [SB3 Contrib: Maskable PPO](https://sb3-contrib.readthedocs.io/en/master/modules/ppo_mask.html)
- [Gymnasium: Taxi](https://gymnasium.farama.org/environments/toy_text/taxi/)
